# 02 - Nettoyage des données

Objectif : préparer des données fiables pour l'analyse des incidents et des KPI de performance.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'incident_event_log.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'data' / 'database'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
df = pd.read_csv(RAW_DATA, na_values=['?'], low_memory=False)
print(df.shape)

(141712, 36)


## Nettoyage des champs clés

On convertit les dates, on remplace les valeurs manquantes utiles pour l'analyse et on extrait les niveaux numériques de priorité, impact et urgence.

In [3]:
date_cols = ['opened_at', 'resolved_at', 'closed_at', 'sys_created_at', 'sys_updated_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d/%m/%Y %H:%M', errors='coerce')

for col in ['category', 'subcategory', 'assignment_group', 'contact_type', 'location']:
    df[col] = df[col].fillna('Inconnu')

for col in ['assigned_to', 'resolved_by', 'opened_by']:
    df[col] = df[col].fillna('Non assigne')

for col in ['reassignment_count', 'reopen_count', 'sys_mod_count']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

if df['made_sla'].dtype != bool:
    df['made_sla'] = df['made_sla'].astype(str).str.lower().map({'true': True, 'false': False}).fillna(False)

def niveau(valeur):
    if pd.isna(valeur):
        return np.nan
    return pd.to_numeric(str(valeur).split(' - ')[0], errors='coerce')

df['priority_level'] = df['priority'].apply(niveau)
df['impact_level'] = df['impact'].apply(niveau)
df['urgency_level'] = df['urgency'].apply(niveau)

print('Nettoyage terminé')

Nettoyage terminé


## Table au niveau incident

Le fichier brut contient plusieurs lignes par incident. Pour les KPI métier, on garde le dernier état connu de chaque incident à partir de `sys_updated_at`.

In [4]:
df_event_clean = df.sort_values(['number', 'sys_updated_at']).reset_index(drop=True)

df_incident = (
    df.sort_values('sys_updated_at')
      .drop_duplicates(subset='number', keep='last')
      .reset_index(drop=True)
)

for col in ['resolution_time_hours', 'closure_time_hours']:
    if col in df_incident.columns:
        df_incident = df_incident.drop(columns=col)

df_incident['resolution_time_hours'] = (
    (df_incident['resolved_at'] - df_incident['opened_at']).dt.total_seconds() / 3600
)
df_incident['closure_time_hours'] = (
    (df_incident['closed_at'] - df_incident['opened_at']).dt.total_seconds() / 3600
)

for col in ['resolution_time_hours', 'closure_time_hours']:
    df_incident.loc[df_incident[col] < 0, col] = np.nan
    df_incident[col] = df_incident[col].round(2)

df_incident['sla_compliant'] = df_incident['made_sla'].astype(int)
df_incident['has_reassignment'] = df_incident['reassignment_count'] > 0
df_incident['has_reopen'] = df_incident['reopen_count'] > 0

df_incident['opened_month'] = df_incident['opened_at'].dt.to_period('M').astype(str)
df_incident['opened_hour'] = df_incident['opened_at'].dt.hour

df_event_clean.shape, df_incident.shape

((141712, 39), (24918, 46))

## Contrôle qualité après nettoyage

Avant d'exporter, on vérifie que la table finale contient bien une ligne par incident et que les durées calculées sont exploitables.

In [5]:
quality_checks = pd.DataFrame({
    'controle': [
        'Incidents dans la table finale',
        'Doublons sur number',
        'Incidents sans date de résolution',
        'Incidents sans date de clôture',
        'Durées de résolution négatives',
        'Durées de clôture négatives'
    ],
    'valeur': [
        len(df_incident),
        df_incident['number'].duplicated().sum(),
        df_incident['resolved_at'].isna().sum(),
        df_incident['closed_at'].isna().sum(),
        (df_incident['resolution_time_hours'] < 0).sum(),
        (df_incident['closure_time_hours'] < 0).sum()
    ]
})
quality_checks

,controle,valeur
0,Incidents dans la table finale,24918
1,Doublons sur number,0
2,Incidents sans date de résolution,1556
3,Incidents sans date de clôture,0
4,Durées de résolution négatives,0
5,Durées de clôture négatives,0


## Export

On exporte deux fichiers : le journal nettoyé complet et la table incident utilisée pour les KPI.

In [6]:
event_path = PROCESSED_DIR / 'incident_event_log_clean.csv'
incident_path = PROCESSED_DIR / 'incident_clean.csv'

df_event_clean.to_csv(event_path, index=False)
df_incident.to_csv(incident_path, index=False)

print(f'Fichier événement : {event_path}')
print(f'Fichier incident : {incident_path}')

Fichier événement : c:\Users\djami\Desktop\Projects\IT-Service-Performance-Analytics\data\processed\incident_event_log_clean.csv
Fichier incident : c:\Users\djami\Desktop\Projects\IT-Service-Performance-Analytics\data\processed\incident_clean.csv


## Conclusion

Les données sont prêtes pour l'analyse : les dates sont normalisées, les valeurs manquantes principales sont traitées et les indicateurs de base sont créés.

La table `incident_clean.csv` contient une seule ligne par incident, correspondant au dernier état connu. Les KPI suivants décrivent donc l'état final des incidents, tandis que `incident_event_log_clean.csv` conserve l'historique détaillé des événements.